# Randomised Benchmarking on ibm_fez actual backend

This notebook implements:
1. Connect to IBM Quantum hardware (ibm_fez)
2. Standard Randomised Benchmarking (RB) to extract depolarizing parameter p
3. Interleaved Randomised Benchmarking (IRB) to characterize specific gates
4. Data analysis

## Setup and Imports

In [1]:
%pip install --quiet qiskit qiskit-aer qiskit-ibm-runtime numpy scipy matplotlib pandas

Note: you may need to restart the kernel to use updated packages.


In [2]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import scipy.optimize
from collections import deque
from qiskit import QuantumCircuit, transpile
from qiskit.quantum_info import Operator
from qiskit_aer import AerSimulator
from qiskit_ibm_runtime import QiskitRuntimeService
import os

print("Imports successful!")

Imports successful!


## 1. Connect to ibm_fez

In [3]:
IBM_QUANTUM_TOKEN = "r4IqbTTUxpez5jP4UIcfKOo-f8xki_93SKdFYCNUASB7"
QiskitRuntimeService.save_account(channel="ibm_quantum_platform",
                                  token="<IBM_QUANTUM_TOKEN>", # Use the 44-character API_KEY you created and saved from the IBM Quantum Platform Home dashboard
                                  instance="<crn:v1:bluemix:public:quantum-computing:us-east:a/622e241fa14f4c2aa91941581327706a:f152fd82-1da7-4a2d-861f-1d5c56136f71::>",
                                  name="open",
                                  overwrite=True)

In [4]:
# Qiskit Runtime migration: use new service initialization (no channel argument), and recommend token via env or save_account
from qiskit_ibm_runtime import QiskitRuntimeService
import os

# Option 1: Set token as environment variable (recommended, do this before running the notebook)
# os.environ["QISKIT_IBM_TOKEN"] = "YOUR_TOKEN"

# Option 2: Save account (only once per environment, then comment out)
# QiskitRuntimeService.save_account(token="YOUR_TOKEN")

# Initialize the service (no channel argument needed)
service = QiskitRuntimeService(channel="ibm_quantum_platform", instance="crn:v1:bluemix:public:quantum-computing:us-east:a/622e241fa14f4c2aa91941581327706a:f152fd82-1da7-4a2d-861f-1d5c56136f71::")

#Backend should be ibm_fez
backend = service.backend("ibm_fez", instance="crn:v1:bluemix:public:quantum-computing:us-east:a/622e241fa14f4c2aa91941581327706a:f152fd82-1da7-4a2d-861f-1d5c56136f71::")
print(f"Connected to {backend.name}")
print(f"Basis gates: {backend.basis_gates}")


Connected to ibm_fez
Basis gates: ['cz', 'id', 'rz', 'sx', 'x']


## 2. Generate Single-Qubit Clifford Group
To reduce usage time, the sequences are generated locally and will be tested later.

In [5]:
def normalize_up_to_global_phase(U):
    """Normalize unitary matrix to remove global phase."""
    idx = np.argmax(np.abs(U))
    phase = np.angle(U.flatten()[idx])
    return U * np.exp(-1j * phase)

def generate_single_qubit_cliffords():
    """Generate all 24 single-qubit Clifford gates using H and S."""
    generating_gates = {'H': lambda qc: qc.h(0), 'S': lambda qc: qc.s(0)}
    seen_matrices = []
    circuits = []
    queue = deque([QuantumCircuit(1)])  #We should start with identity
    
    while queue:
        qc = queue.popleft()
        U = Operator(qc).data
        U_norm = normalize_up_to_global_phase(U)
        
        if any(np.allclose(U_norm, M) for M in seen_matrices):
            continue
        seen_matrices.append(U_norm)
        circuits.append(qc)
        
        for gate_func in generating_gates.values():
            new_qc = qc.copy()
            gate_func(new_qc)
            queue.append(new_qc)
    
    assert len(circuits) == 24, f"Expected 24 Cliffords, found {len(circuits)}"
    return circuits

#Generate Clifford gates and compute the inverse
cliffords_list = generate_single_qubit_cliffords()
cliffords_matrices = [normalize_up_to_global_phase(Operator(qc).data) for qc in cliffords_list]
I2 = np.eye(2, dtype=complex)

cliffords_inverses = {}
for i, U_i in enumerate(cliffords_matrices):
    for j, U_j in enumerate(cliffords_matrices):
        product_norm = normalize_up_to_global_phase(U_i @ U_j)
        if np.allclose(product_norm, I2):
            cliffords_inverses[i] = j
            break

print(f"Generated {len(cliffords_list)} Clifford circuits")
print(f"Computed inverse mapping for all Cliffords")

Generated 24 Clifford circuits
Computed inverse mapping for all Cliffords


## 3. Create RB/IRB Sequence Function

In [6]:
def create_rb_sequence(clifford_circuits, cliffords_matrices, m, target_gate_circuit=None, seed=None):
    """
    Create an RB or IRB sequence.
    
    Args:
        clifford_circuits: List of 24 Clifford QuantumCircuits
        cliffords_matrices: List of normalized Clifford unitary matrices
        m: Sequence length (number of random Cliffords)
        target_gate_circuit: Gate to interleave (None for standard RB)
        seed: Random seed for reproducibility
    
    Returns:
        QuantumCircuit with measurement
    
    Sequence structure:
        Standard RB:  C_1, C_2, ..., C_m, C_inverse, measure
        Interleaved:  C_1, G, C_2, G, ..., C_m, G, C_inverse, measure
    """
    rng = np.random.default_rng(seed)
    indices = rng.choice(len(clifford_circuits), size=m, replace=True)
    
    qc = QuantumCircuit(1, 1) #We only need 1 qb to test single qb gates
    
    #Apply random Clifford gates (and optionally interleave target gate)
    for idx in indices:
        qc.compose(clifford_circuits[idx], inplace=True)
        if target_gate_circuit is not None:
            qc.compose(target_gate_circuit, inplace=True)
    
    #Compute inverse Clifford that returns to |0⟩
    U_fwd = Operator(qc).data
    U_fwd_norm = normalize_up_to_global_phase(U_fwd)
    I = np.eye(2, dtype=complex)
    
    inverse_idx = None
    for j, U_j in enumerate(cliffords_matrices):
        product = normalize_up_to_global_phase(U_fwd_norm @ U_j)
        if np.allclose(product, I):
            inverse_idx = j
            break
    
    if inverse_idx is None:
        raise RuntimeError("Could not find inverse Clifford")
    
    qc.compose(clifford_circuits[inverse_idx], inplace=True)
    qc.measure(0, 0)
    return qc

print("RB/IRB sequence function defined")

RB/IRB sequence function defined


In [7]:
#Helper: extract gate sequence from a circuit and store as tuple, we they can be visualised later (last cell)

def extract_gate_sequence(qc, include_measure=False):
    """Return a tuple of gate operation names in order.
    Set include_measure=True to include 'measure' operations."""
    names = []
    for instruction, qargs, cargs in qc.data:
        name = instruction.name
        if not include_measure and name == 'measure':
            continue
        names.append(name)
    return tuple(names)


## 4. Standard RB - Extract Depolarizing Parameter p

In [ ]:
#Standard RB: No target gate interleaving (using Qiskit Runtime Sampler)
from qiskit_ibm_runtime import Sampler

sequence_lengths = [2, 4, 6, 8] # Removed m=12 to speed up execution
num_sequences = 15 #In some papers, 40 is used, but it is necessary to reduce time usage
shots = 100 #Some papers used 100 runs
seed_base = 54321

survival_probs_rb = [] # To store average survival probabilities

print("Running Standard RB on ibm_fez backend (Sampler primitive)...")
sampler = Sampler(backend)
for m in sequence_lengths:
    probs = []
    for k in range(num_sequences):
        seed = seed_base + m * 1000 + k
        rb_seq = create_rb_sequence(
            clifford_circuits=cliffords_list,
            cliffords_matrices=cliffords_matrices,
            m=m,
            target_gate_circuit=None,  # No interleaving
            seed=seed
        )
        rb_seq_transpiled = transpile(rb_seq, backend=backend)
        #Sampler primitive to run the circuit
        job = sampler.run([rb_seq_transpiled], shots=shots)
        result = job.result()
        print("DEBUG: result object:", result)
        try:
            # Robust extraction for different Qiskit result types and key types
            quasi_dist = None
            if hasattr(result, "quasi_dists"):
                quasi_dist = result.quasi_dists[0]
            elif hasattr(result, "quasi_distribution"):
                quasi_dist = result.quasi_distribution
                if isinstance(quasi_dist, list):
                    quasi_dist = quasi_dist[0]
            elif isinstance(result, (list, tuple)) and len(result) > 0:
                quasi_dist = result[0]
            elif isinstance(result, dict) and 0 in result:
                quasi_dist = result
            print("DEBUG: quasi_dist:", quasi_dist)
            if quasi_dist is not None:
                print("DEBUG: quasi_dist keys:", list(quasi_dist.keys()) if hasattr(quasi_dist, 'keys') else None)
                prob_0 = quasi_dist.get(0, quasi_dist.get('0', 0.0))
            else:
                prob_0 = 0.0
        except Exception as e:
            print("Could not extract probability from result:", e)
            prob_0 = 0.0
        probs.append(prob_0)
    avg_prob = np.mean(probs)
    survival_probs_rb.append(avg_prob)
    print(f"  m={m:2d}: F_seq = {avg_prob:.6f}")

print("Standard RB complete (ibm_fez, Sampler)")

Running Standard RB on ibm_fez backend (Sampler primitive)...
  m= 2: F_seq = 0.000000
  m= 4: F_seq = 0.000000
  m= 6: F_seq = 0.000000
  m= 8: F_seq = 0.000000
Standard RB complete (ibm_fez, Sampler)


In [12]:
#Zero-order Standard RB Fit: F_seq(m) = A * p^m + B (0th order)
m_vals = np.array(sequence_lengths)
F_seq_rb = np.array(survival_probs_rb)

def fit_func(m, A, p, B):
    return A * p**m + B

params_rb, _ = scipy.optimize.curve_fit(
    fit_func, m_vals, F_seq_rb,
    p0=[0.95, 0.98, 0.05],
    bounds=([0.1, 0.85, 0.001], [1, 0.9999, 0.3])
)
A_rb, p_rb, B_rb = params_rb
r_rb = (2 - 1) / 2 * (1 - p_rb)  #Average error per Clifford gate

print(f"\n=== Standard RB Fit (0th order) ===")
print(f"Fit: F_seq(m) = {A_rb:.6f} × {p_rb:.8f}^m + {B_rb:.6f}")
print(f"Depolarizing parameter p: {p_rb:.8f}")
print(f"Average error per Clifford r: {r_rb:.8f} ({r_rb*100:.6f}%)")
print(f"Gate fidelity: {(1-r_rb)*100:.6f}%")


=== Standard RB Fit (0th order) ===
Fit: F_seq(m) = 0.100000 × 0.85000001^m + 0.001000
Depolarizing parameter p: 0.85000001
Average error per Clifford r: 0.07500000 (7.500000%)
Gate fidelity: 92.500000%


In [ ]:
#First-order RB fit: F_seq^(1)(m) = A1 * p^m + C1 * (m-1) * p^(m-2) + B1

def fit_func_first_order(m, A1, p1, C1, B1):
    m = np.asarray(m, dtype=float)
    return A1 * (p1 ** m) + C1 * (m - 1) * (p1 ** (m - 2)) + B1

try:
    params_rb_fo, _ = scipy.optimize.curve_fit(
        fit_func_first_order,
        m_vals,
        F_seq_rb,
        p0=[0.5, 0.99, 0.01, 0.5],
        bounds=([0.0, 0.9, -1.0, 0.0], [1.0, 1.0, 1.0, 1.0])
    )
    A1_rb, p1_rb, C1_rb, B1_rb = params_rb_fo
    r1_rb = (2 - 1) / 2 * (1 - p1_rb)
    print("\n=== Standard RB Fit (1st order) ===")
    print(f"Fit: F_seq^(1)(m) = {A1_rb:.6f} * {p1_rb:.8f}^m + {C1_rb:.6f} * (m-1) * {p1_rb:.8f}^(m-2) + {B1_rb:.6f}")
    print(f"Depolarizing parameter p (first-order): {p1_rb:.8f}")
    print(f"Average error per Clifford r (first-order): {r1_rb:.8f} ({r1_rb*100:.6f}%)")
except Exception as e:
    print(f"First-order RB fit failed: {e}")

## 5. Interleaved RB - Characterize RX Gate

In [ ]:
#Define target gate (RZ(pi/2) gate)
target_RZ = QuantumCircuit(1)
target_RZ.rz(np.pi/2, 0)
"""
These two lines can be updated according to the we wish to test. For:
 X: target_X = QuantumCircuit(1) , target_X.x(0)
SX: target_SX = QuantumCircuit(1) , target_SX.sx(0)
RX: target_RX = QuantumCircuit(1) , target_RX.rx(np.pi/2, 0)
RZ: target_RZ = QuantumCircuit(1) , target_RZ.rz(np.pi/2, 0)
"""
#Interleaved RB: RZ(pi/2) gate after each Clifford
from qiskit_ibm_runtime import Sampler
survival_probs_irb = []

print("Running Interleaved RB (RZ(pi/2) gate) on ibm_fez backend (Sampler primitive)...")
sampler_irb = Sampler(backend)
for m in sequence_lengths:
    probs = []
    for k in range(num_sequences):
        seed = seed_base + m * 1000 + k + 99999
        irb_seq = create_rb_sequence(
            clifford_circuits=cliffords_list,
            cliffords_matrices=cliffords_matrices,
            m=m,
            target_gate_circuit=target_RZ,  # Interleave RZ(pi/2) gate
            seed=seed
        )
        irb_seq_transpiled = transpile(irb_seq, backend=backend)
        # Use Sampler primitive to run the circuit
        job = sampler_irb.run([irb_seq_transpiled], shots=shots)
        result = job.result()
        
        try:
            # For Qiskit Runtime Sampler (newer versions)
            quasi_dist = result.quasi_dists[0]
            prob_0 = quasi_dist.get(0, 0.0)
        except AttributeError:
            # For other result types (e.g., get_counts)
            counts = result.get_counts()
            prob_0 = counts.get('0', 0) / shots
        probs.append(prob_0)
    avg_prob = np.mean(probs)
    survival_probs_irb.append(avg_prob)
    print(f"  m={m:2d}: F_seq = {avg_prob:.6f}")

print("Interleaved RB complete (ibm_fez, Sampler)")

In [ ]:
#Fit Interleaved RB data (RZ)
F_seq_irb = np.array(survival_probs_irb)

params_irb, _ = scipy.optimize.curve_fit(
    fit_func, m_vals, F_seq_irb,
    p0=[0.95, 0.98, 0.05],
    bounds=([0.1, 0.85, 0.001], [1, 0.9999, 0.3])
)
A_irb, p_irb, B_irb = params_irb
r_irb = (2 - 1) / 2 * (1 - p_irb)

#Calculate RZ gate error from comparison with standard RB
#Formula: p_irb = p_rb * p_RZ (approximately)
p_RZ = min(p_irb / p_rb if p_rb > 0 else 1.0, 1.0)  # Clamp to [0, 1]
r_RZ = (2 - 1) / 2 * (1 - p_RZ)

"""
Alter gate names in the print!!!
"""
print(f"\n=== Interleaved RB Fit (0th order) (RZ Gate) ===") 
print(f"Fit: F_seq(m) = {A_irb:.6f} × {p_irb:.8f}^m + {B_irb:.6f}")
print(f"Depolarizing parameter p_irb: {p_irb:.8f}")
print(f"Average error per (Clifford+RZ) r_irb: {r_irb:.8f} ({r_irb*100:.6f}%)") 
print(f"\nRZ gate specific:") 
print(f"  p_RZ = p_irb / p_rb = {p_RZ:.8f}") 
print(f"  RZ gate error r_RZ: {r_RZ:.8f} ({r_RZ*100:.6f}%)") 
print(f"  RZ gate fidelity: {(1-r_RZ)*100:.6f}%") 

In [ ]:
# First-order IRB fit: F_seq^(1)(m) = A1 * p^m + C1 * (m-1) * p^(m-2) + B1 (RZ)
try:
    params_irb_fo, _ = scipy.optimize.curve_fit(
        fit_func_first_order,
        m_vals,
        F_seq_irb,
        p0=[0.5, 0.99, 0.01, 0.5],
        bounds=([0.0, 0.9, -1.0, 0.0], [1.0, 1.0, 1.0, 1.0])
    )
    A1_irb, p1_irb, C1_irb, B1_irb = params_irb_fo
    r1_irb = (2 - 1) / 2 * (1 - p1_irb)
    # Gate-specific first-order parameter via p_RZ1 ≈ p1_irb / p1_rb
    p1_RZ = min(p1_irb / p1_rb if p1_rb > 0 else 1.0, 1.0)  # Clamp to [0, 1]
    r1_RZ = (2 - 1) / 2 * (1 - p1_RZ)
    print("\n=== Interleaved RB Fit (1st order) (RZ Gate) ===")
    print(f"Fit: F_seq^(1)(m) = {A1_irb:.6f} * {p1_irb:.8f}^m + {C1_irb:.6f} * (m-1) * {p1_irb:.8f}^(m-2) + {B1_irb:.6f}")
    print(f"Depolarizing parameter p_irb (first-order): {p1_irb:.8f}")
    print(f"Average error per (Clifford+RZ) r_irb (first-order): {r1_irb:.8f} ({r1_irb*100:.6f}%)")
    print(f"\nRZ gate (first-order): p_RZ1 = p1_irb / p1_rb = {p1_RZ:.8f}")
    print(f"RZ gate error r_RZ1: {r1_RZ:.8f} ({r1_RZ*100:.6f}%) | Fidelity: {(1-r1_RZ)*100:.6f}%")
except Exception as e:
    print(f"First-order IRB fit failed: {e}")

## 6. Visualization and Data Summary

In [ ]:
# Plot Standard RB and Interleaved RB comparison (RZ)
plt.figure(figsize=(12, 5))

# Standard RB plot
plt.subplot(1, 2, 1)
plt.plot(m_vals, F_seq_rb, 'o', markersize=8, label='Standard RB data', color='yellowgreen')
plt.plot(m_vals, fit_func(m_vals, *params_rb), '-', linewidth=2, label=f'Fit: p={p_rb:.5f}', color='yellowgreen')
plt.xlabel('Sequence length m', fontsize=12)
plt.ylabel('Sequence fidelity $F_{seq}(m)$', fontsize=12)
plt.title('Standard RB (Clifford-only)', fontsize=13, fontweight='bold')
plt.legend(fontsize=10)
plt.grid(True, alpha=0.3)

# Interleaved RB plot (RZ)
plt.subplot(1, 2, 2)
plt.plot(m_vals, F_seq_irb, 's', markersize=8, label='Interleaved RB data', color='deeppink')
plt.plot(m_vals, fit_func(m_vals, *params_irb), '-', linewidth=2, label=f'Fit: p={p_irb:.5f}', color='deeppink')
plt.xlabel('Sequence length m', fontsize=12)
plt.ylabel('Sequence fidelity $F_{seq}(m)$', fontsize=12)
plt.title('Interleaved RB (RZ gate)', fontsize=13, fontweight='bold')
plt.legend(fontsize=10)
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
#Shows one of the actual sequences used in the RB/IRB runs (same seeds as above)
#m_view and k_view can be adjusted to show different sequences; interleaved=True shows RZ-gate IRB.
m_view = 2         #sequence_lengths
k_view = 0          #0-based index into the num_sequences draws
interleaved = False #False = Standard RB, True = Interleaved RB (RZ gate)

if m_view not in sequence_lengths:
    raise ValueError(f"m_view {m_view} not in sequence_lengths {sequence_lengths}")
if not (0 <= k_view < num_sequences):
    raise ValueError(f"k_view {k_view} must be between 0 and {num_sequences-1}")

seed_offset = 99999 if interleaved else 0
seed = seed_base + m_view * 1000 + k_view + seed_offset

seq = create_rb_sequence(
    clifford_circuits=cliffords_list,
    cliffords_matrices=cliffords_matrices,
    m=m_view,
    target_gate_circuit=(target_RZ if interleaved else None),
    seed=seed,
)

seq_label = "Interleaved RB (RZ gate)" if interleaved else "Standard RB"
seq_gate_tuple = extract_gate_sequence(seq)

print(f"{seq_label} — actual sequence used during run")
print(f"m = {m_view}, sequence index k = {k_view}, seed = {seed}")
print(f"Gate tuple: {seq_gate_tuple}")
print(f"Circuit depth: {seq.depth()}")
print(f"Number of gates (incl. measure): {len(seq.data)}")

%matplotlib inline
fig = seq.draw('mpl')
plt.show()

In [ ]:
#Debug Fit order 1
print("m_vals:", m_vals)
print("F_seq_irb:", F_seq_irb)
print("p0:", [0.5, 0.99, 0.01, 0.5])
print("bounds:", ([0.0, 0.9, -1.0, 0.0], [1.0, 1.0, 1.0, 1.0]))

In [19]:
#DEBUG
from qiskit.circuit import (
    Parameter,
    QuantumCircuit,
    ClassicalRegister,
    QuantumRegister,
)
from qiskit.transpiler import generate_preset_pass_manager
from qiskit.quantum_info import SparsePauliOp
from qiskit.primitives.containers import BitArray
 
from qiskit_ibm_runtime import (
    QiskitRuntimeService,
    EstimatorV2 as Estimator,
    SamplerV2 as Sampler,
)
 
import numpy as np
 
# Instantiate runtime service and get
# the least busy backend
service = QiskitRuntimeService()
backend = service.least_busy(operational=True, simulator=False)
 
# Define a circuit with two parameters.
circuit = QuantumCircuit(2)
circuit.h(0)
 
# Transpile the circuit
pm = generate_preset_pass_manager(optimization_level=1, backend=backend)
transpiled_circuit = pm.run(circuit)
layout = transpiled_circuit.layout
 
# Now define a sweep over parameter values, the last axis of dimension 2 is
# for the two parameters "a" and "b"
params = np.vstack(
    [
        np.linspace(-np.pi, np.pi, 100),
        np.linspace(-4 * np.pi, 4 * np.pi, 100),
    ]
).T
 
# Define three observables. The inner length-1 lists cause this array of
# observables to have shape (3, 1), rather than shape (3,) if they were
# omitted.
observables = [
    [SparsePauliOp(["XX", "IY"], [0.5, 0.5])],
    [SparsePauliOp("XX")],
    [SparsePauliOp("IY")],
]
# Apply the same layout as the transpiled circuit.
observables = [
    [observable.apply_layout(layout) for observable in observable_set]
    for observable_set in observables
]
 
# Estimate the expectation value for all 300 combinations of observables
# and parameter values, where the pub result will have shape (3, 100).
#
# This shape is due to our array of parameter bindings having shape
# (100, 2), combined with our array of observables having shape (3, 1).
estimator_pub = (transpiled_circuit, observables, params)
 
# Instantiate the new estimator object, then run the transpiled circuit
# using the set of parameters and observables.
estimator = Estimator(mode=backend)
job = estimator.run([estimator_pub])
result = job.result()

qiskit_runtime_service.__init__:WARNING:2026-01-19 00:50:29,466: Instance was not set at service instantiation. Free and trial plan instances will be prioritized. Based on the following filters: (tags: None, region: us-east, eu-de), and available plans: (open), the available account instances are: open-instance. If you need a specific instance set it explicitly either by using a saved account with a saved default instance or passing it in directly to QiskitRuntimeService().
qiskit_runtime_service.backends:WARNING:2026-01-19 00:50:29,836: Loading instance: open-instance, plan: open
qiskit_runtime_service.backends:WARNING:2026-01-19 00:50:31,236: Using instance: open-instance, plan: open


ValueError: Length of () inconsistent with last dimension of [[ -3.14159265 -12.56637061]
 [ -3.07812614 -12.31250454]
 [ -3.01465962 -12.05863847]
 [ -2.9511931  -11.8047724 ]
 [ -2.88772658 -11.55090632]
 [ -2.82426006 -11.29704025]
 [ -2.76079354 -11.04317418]
 [ -2.69732703 -10.7893081 ]
 [ -2.63386051 -10.53544203]
 [ -2.57039399 -10.28157596]
 [ -2.50692747 -10.02770988]
 [ -2.44346095  -9.77384381]
 [ -2.37999443  -9.51997774]
 [ -2.31652792  -9.26611167]
 [ -2.2530614   -9.01224559]
 [ -2.18959488  -8.75837952]
 [ -2.12612836  -8.50451345]
 [ -2.06266184  -8.25064737]
 [ -1.99919533  -7.9967813 ]
 [ -1.93572881  -7.74291523]
 [ -1.87226229  -7.48904915]
 [ -1.80879577  -7.23518308]
 [ -1.74532925  -6.98131701]
 [ -1.68186273  -6.72745093]
 [ -1.61839622  -6.47358486]
 [ -1.5549297   -6.21971879]
 [ -1.49146318  -5.96585272]
 [ -1.42799666  -5.71198664]
 [ -1.36453014  -5.45812057]
 [ -1.30106362  -5.2042545 ]
 [ -1.23759711  -4.95038842]
 [ -1.17413059  -4.69652235]
 [ -1.11066